# 01 · 数据清洗、EDA 与数据质量

**负责人** 成员 2  
**输入** 本地 UTF-8 TSV 或 CSV，默认 `data/raw/Restaurant_Reviews.tsv`，文本列 `Review`，标签列 `Liked`（0 负面，1 正面）。数据版本见 [数据字典](../data/data_dictionary.md)。  
**输出** `data/processed/` 下的清洗结果、训练集、测试集和划分清单，以及 `results/` 下的数据质量审计、统计表和英文图表。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from restaurant_sentiment.config import PROJECT_ROOT
from restaurant_sentiment.eda import run_eda

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Restaurant_Reviews.tsv"
TEXT_COLUMN, LABEL_COLUMN = "Review", "Liked"
TABLES = PROJECT_ROOT / "results" / "tables"
FIGURES = PROJECT_ROOT / "results" / "figures"

audit = run_eda(Path(DATA_PATH), TEXT_COLUMN, LABEL_COLUMN)
print(f"Input: {audit['source_file']}")
print(f"SHA-256: {audit['source_sha256']}")
print(f"Split seed: {audit['random_state']} | Test fraction: {audit['test_size']:.0%}")

## 数据质量与清洗

使用 NFC 统一 Unicode 表示，去除嵌入的 BOM，将控制字符和零宽空格转成空格，再合并空白。保留大小写、标点、重音和否定表达。删除空评论和缺标签记录。将支持的标签写法映射为整数 0 和 1，遇到其他标签时停止并提示具体值。

按规范化后的完整文本检查重复。标签一致的评论保留首次出现的记录，标签冲突的同文组全部移除。审计表的缺失、字符和原始重复项可能重叠，`removed_*` 是按顺序记账的互斥移除项。

In [ ]:
quality = pd.read_csv(TABLES / "data_quality.csv")
display(quality)
counts = audit["counts"]
removed = sum(value for key, value in counts.items() if key.startswith("removed_"))
assert counts["input_rows"] == counts["retained_rows"] + removed
assert audit["train_test_exact_review_overlap"] == 0

## 类别分布与统一划分

按标签分层做一次 80/20 划分，使用 `random_state=8017`。后续 notebook 读取 `data/processed/train.csv` 和 `data/processed/test.csv`。两个文件的 `review_id` 对应清洗后顺序，从 1 开始。TF-IDF 与调参在训练数据内完成。

In [ ]:
distribution = pd.read_csv(TABLES / "class_distribution.csv")
display(distribution.style.format({"share": "{:.2%}"}))
display(Image(filename=str(FIGURES / "eda_class_distribution.png"), width=610))

## 评论长度

用清洗后的样本计算字符数与词数。字符数包含空格和标点，词数按字母序列统计，内部含撇号的缩写算一个词。直方图使用相同组距和坐标范围，虚线标出各类的中位数。

In [ ]:
lengths = pd.read_csv(TABLES / "review_lengths.csv")
display(lengths.style.format(precision=1))
display(Image(filename=str(FIGURES / "eda_review_lengths.png"), width=610))

## 训练集高频词

按包含该词的训练评论占同类评论的比例排序。每条评论内重复出现的词只计一次，表中另外保留总出现次数。分词后按小写统计，过滤常见英文停用词并保留否定词和带撇号的缩写。词频用来描述语料和检查清洗规则。

In [ ]:
terms = pd.read_csv(TABLES / "training_term_frequencies.csv")
display(terms.style.format({"document_share": "{:.1%}"}))
display(Image(filename=str(FIGURES / "eda_training_terms.png"), width=820))